# 04 · SMLP4Rec + destination prior + sameDest (hybrid)

Thêm tín hiệu lịch sử cùng destination vào SMLP4Rec + prior của notebook 03. Cùng checkpoint, cách chia, prior và tập candidate; không huấn luyện lại.

**sameDest**: user thường đặt lại một cluster mà họ đã đặt tại destination đang tìm. `sameDest(k)` = tổng `0.7^age` trên các booking trước đó của user vào cluster k tại destination truy vấn (age 0 = gần nhất), chuyển thành phân phối trên các cluster.

**Công thức** (`nbp.hybrid.fusion`): mọi thành phần là phân phối trên K = 100 cluster, trộn với phân phối đều trước khi lấy log (tích của các expert đã làm mượt):

```
score(k) = log q_SMLP4Rec(k) + w_p * log q_prior(k) + w_s * log q_sameDest(k)
log q(k) = log((1 - alpha) * p(k) + alpha / K)        one alpha for all three components
p_sameDest(k) = sameDest(k) / sum_j sameDest(j)        (0.7^age recency, query destination only)
              = 1/K when the user has no booking at the query destination (constant: no effect)
```

Mỗi số hạng bị chặn trong cùng một khoảng, nên các trọng số đọc như độ tin cậy tương đối, và số hạng sameDest tự tắt ở các truy vấn destination mới.

**Quy tắc lựa chọn (chỉ dùng valid; test đọc một lần ở cuối):**

1. Mục tiêu mỗi dòng: `Recall@5 + NDCG@5` (cả hai metric chính, gộp thành một số).
2. `alpha`, `w_p`, `w_s` trên một lưới có giới hạn (trọng số ≤ 3, SMLP4Rec cố định 1). **Quy tắc một sai số chuẩn** (Breiman et al. 1984; Hastie, Tibshirani & Friedman, ESL §7.10): trong các cấu hình có mục tiêu trung bình nằm trong một sai số chuẩn của cấu hình tốt nhất (SE của chênh lệch theo cặp từng dòng, gom cụm theo user), chọn cấu hình đơn giản nhất = trọng số lớn nhất nhỏ nhất, rồi tổng trọng số nhỏ nhất, rồi alpha lớn nhất (làm mượt nhiều hơn).
3. Cohort: các phương án ứng viên từ 1 ô (global) đến 4 ô, chỉ xây từ các mẫu cốt lõi có độ phủ rộng (độ dài lịch sử, destination truy vấn đã có trong lịch sử). Mỗi phương án được chấm **ngoài fold** (CV 2-fold cấp user trên valid, 3 seed chia fold; trọng số từng ô chọn bằng quy tắc 2 trên nửa tinh chỉnh). Quy tắc một-SE sau đó chọn phương án có ít ô nhất mà mục tiêu ngoài fold nằm trong một SE của phương án tốt nhất.
4. Dòng cold (L = 0): chỉ prior, như notebook 03.

**Tham chiếu**: SMLP4Rec + prior của notebook 03 (`log p_SMLP4Rec + w_p * log(p_prior + 1e-6)`, `w_p` global chọn trên valid ở đó), được chấm lại từ cùng các thành phần.

Kernel: `smlp4rec`. Cần `data/interim/recbole/expedia_dest/expedia_dest.inter` và `train.csv` thô để tính prior.

In [1]:
import functools
import itertools
import json
import logging
import sys
import time
from pathlib import Path

import numpy as np
import torch  # before pandas (Windows c10.dll issue)

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
torch.load = functools.partial(torch.load, weights_only=False)

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from nbp.eval.bootstrap import paired_bootstrap
from nbp.eval.metrics import summarize, target_rank
from nbp.hybrid.fusion import cluster_se, one_se_pick, same_dest_distribution, smoothed_log
from nbp.hybrid.same_dest import same_dest_scores
from src.models.smlprec import SMLPREC

logging.disable(logging.INFO)

CONFIG = ROOT / "configs" / "smlprec_expedia.yaml"
WORK = ROOT / "data" / "interim" / "recbole"
EVENT_COLS = ROOT / "data" / "interim" / "expedia_event_cols_clean.npz"  # row position = event index = TIME
INTER_DEST = WORK / "expedia_clean_dest" / "expedia_clean_dest.inter"
REF_DIR = ROOT / "results" / "week4_rebuild" / "clean_chain"  # outputs and predecessor results on the collapsed data
OLD_DIR = ROOT / "results" / "week3_implementation"  # same chain on the uncollapsed data (reference only)
REF_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = max((ROOT / "experiments").glob("*_expedia_smlp4rec_clean-plain"))

N_CLUSTERS = 100
M_SMOOTH = 5
EPS = 1e-6  # prior floor of notebook 03 (reference fusion and cold rows)
TIME_MARGIN = 0  # TIME is the exact event index: no float32 margin
RECENCY = 0.7
ALPHAS = [0.05, 0.1, 0.2, 0.3, 0.5]  # uniform-mixing share, one value for all three components
W_GRID = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0]  # w_p and w_s, SMLP4Rec fixed at 1
CV_SEEDS = [0, 1, 2]  # user-level 2-fold CV on valid for the cohort schemes
BUCKETS = [("L=1", 1, 1), ("L=2-4", 2, 4), ("L=5-9", 5, 9), ("L>=10", 10, 10**9)]
LF = json.loads((REF_DIR / "smlprec_expedia_late_fusion.json").read_text(encoding="utf-8"))
FUSION_W = LF["best_global_w_on_valid"]  # notebook 03
print("run dir:", RUN_DIR)

run dir: C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-08_expedia_smlp4rec_clean-plain


## 1. Dữ liệu, prior và các thành phần điểm (giống notebook 03)

In [2]:
config = Config(
    model=SMLPREC,
    dataset="expedia_clean_dest",
    config_file_list=[str(CONFIG)],
    config_dict={
        "data_path": str(WORK),
        "checkpoint_dir": str(RUN_DIR),
        "load_col": {"inter": ["user_id", "item_id", "timestamp", "srch_destination_id"]},
    },
)
init_seed(config["seed"], config["reproducibility"])
t0 = time.time()
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
tok2id = dataset.field2token_id[dataset.iid_field]
cluster_ids = np.array(
    [tok2id[str(k)] for k in range(N_CLUSTERS)]
)  # model column of each raw cluster
col_of_model_id = {int(m): k for k, m in enumerate(cluster_ids)}
ITEM_SEQ = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
ITEM_SEQ_LEN = config["ITEM_LIST_LENGTH_FIELD"]
TIME = config["TIME_FIELD"]


def split_arrays(loader) -> dict:
    d = loader.dataset.inter_feat
    seq, length = d[ITEM_SEQ].numpy(), d[ITEM_SEQ_LEN].numpy()
    y_model = d[config["ITEM_ID_FIELD"]].numpy()
    q_dest = d["srch_destination_id"].numpy().astype(np.int64)
    h_dest = d["srch_destination_id" + config["LIST_SUFFIX"]].numpy().astype(np.int64)
    pos = np.arange(seq.shape[1])[None, :] < length[:, None]
    return {
        "seq": seq,
        "seq_t": d[ITEM_SEQ],
        "length": length,
        "length_t": d[ITEM_SEQ_LEN],
        "y": np.array([col_of_model_id[int(t)] for t in y_model]),  # raw cluster index 0..99
        "user": d[config["USER_ID_FIELD"]].numpy(),
        "old": ((seq == y_model[:, None]) & pos).any(1),
        "q_dest": q_dest,
        "h_dest": h_dest,
        "q_seen": ((h_dest == q_dest[:, None]) & pos).any(1),
    }


arr = {"valid": split_arrays(valid_data), "test": split_arrays(test_data)}
bucket_masks = {
    sp: {b: (a["length"] >= lo) & (a["length"] <= hi) for b, lo, hi in BUCKETS}
    for sp, a in arr.items()
}
print({sp: len(a["y"]) for sp, a in arr.items()}, f"{time.time() - t0:.0f}s")

{'valid': 213869, 'test': 213869} 123s


In [3]:
import pandas as pd  # after torch

t_train_max = float(train_data.dataset.inter_feat[TIME].max())
t_valid_max = float(valid_data.dataset.inter_feat[TIME].max())
cut = t_train_max - TIME_MARGIN


def build_prior_tables(cut_event: float) -> dict:
    """Cluster count tables from the bookings strictly before event index cut_event (collapsed data)."""
    z, k = np.load(EVENT_COLS), int(cut_event)
    b = pd.DataFrame({"srch_destination_id": z["dest"][:k], "hotel_market": z["market"][:k], "hotel_cluster": z["item"][:k]}).astype("int64")

    def table(key):
        t = b.groupby([key, "hotel_cluster"]).size().unstack(fill_value=0)
        return t.reindex(columns=range(N_CLUSTERS), fill_value=0)

    dest, mkt = table("srch_destination_id"), table("hotel_market")
    dm = b.groupby(["srch_destination_id", "hotel_market"]).size().reset_index(name="n")
    modal = dm.sort_values("n", ascending=False).drop_duplicates("srch_destination_id")
    glob = np.bincount(b["hotel_cluster"].values, minlength=N_CLUSTERS).astype(np.float64)
    return {
        "n_bookings": len(b),
        "dest": {int(d): r.astype(np.float64) for d, r in zip(dest.index, dest.values)},
        "market": {int(m): r.astype(np.float64) for m, r in zip(mkt.index, mkt.values)},
        "dest2mkt": dict(
            zip(modal["srch_destination_id"].astype(int), modal["hotel_market"].astype(int))
        ),
        "global": glob / glob.sum(),
    }


def prior_vector(d: int) -> np.ndarray:
    """p(cluster | destination), shape (100,), raw cluster order."""
    row = tables["dest"].get(d)
    if row is None:
        return tables["global"]
    pm = tables["market"][tables["dest2mkt"][d]]
    pm = pm / pm.sum()
    return (row + M_SMOOTH * pm) / (row.sum() + M_SMOOTH)


def prior_prob(dests: np.ndarray) -> np.ndarray:
    """(n, 100) prior distribution of each query destination, raw cluster order."""
    uniq, inv = np.unique(dests, return_inverse=True)
    return np.stack([prior_vector(int(d)) for d in uniq]).astype(np.float32)[inv]


t1 = time.time()
tables = build_prior_tables(cut)
print(
    f"prior cut (unix s) {cut:.0f}; {tables['n_bookings']} bookings, {len(tables['dest'])} destinations; {time.time() - t1:.0f}s"
)

model = SMLPREC(config, dataset)
model.load_state_dict(torch.load(RUN_DIR / "best.pth")["state_dict"])


@torch.no_grad()
def model_prob(a: dict, batch: int = 8192) -> np.ndarray:
    """(n, 100) softmax of the model over the 100 clusters (padding column excluded), raw cluster order."""
    model.eval()
    out = []
    for i in range(0, len(a["y"]), batch):
        s = model.full_sort_predict(
            {
                model.ITEM_SEQ: a["seq_t"][i : i + batch],
                model.ITEM_SEQ_LEN: a["length_t"][i : i + batch],
            }
        )
        s[:, 0] = float("-inf")
        out.append(torch.softmax(s, dim=1).numpy()[:, cluster_ids])
    return np.concatenate(out)


p_model, p_prior, sd = {}, {}, {}
for sp, a in arr.items():
    p_model[sp], p_prior[sp] = model_prob(a), prior_prob(a["q_dest"])
    sd[sp] = same_dest_scores(
        a["seq"],
        a["length"],
        a["h_dest"],
        a["q_dest"],
        dataset.item_num,
        base=RECENCY,
        normalize=True,
    )[:, cluster_ids]
print("components ready", {sp: p_model[sp].shape for sp in arr})

prior cut (unix s) 2424705; 2424705 bookings, 34507 destinations; 1s


components ready {'valid': (213869, 100), 'test': (213869, 100)}


## 2. Tham chiếu: SMLP4Rec + prior (notebook 03)

Được chấm lại từ các thành phần với trọng số global đã chọn của notebook 03 (không tinh chỉnh lại); phải tái tạo số test của notebook 03.

In [4]:
def rank(scores: np.ndarray, y: np.ndarray) -> np.ndarray:
    return target_rank(scores, y)


REF = f"SMLP4Rec + prior (notebook 03, w_p = {FUSION_W})"
ranks = {sp: {} for sp in arr}
for sp in arr:
    ranks[sp][REF] = rank(np.log(p_model[sp]) + FUSION_W * np.log(p_prior[sp] + EPS), arr[sp]["y"])
want = LF["variants"]["test"][f"fusion_global_w={FUSION_W}"]["all"]
got = summarize(ranks["test"][REF])
print(
    f"{REF}: test Recall@5 {got['recall@5']} Recall@10 {got['recall@10']} NDCG@10 {got['ndcg@10']}"
    f" | notebook 03 {want['recall@5']} {want['recall@10']} {want['ndcg@10']}"
)
assert all(got[m] == want[m] for m in ("recall@5", "recall@10", "ndcg@10"))

SMLP4Rec + prior (notebook 03, w_p = 1.5): test Recall@5 0.5851 Recall@10 0.7617 NDCG@10 0.4712 | notebook 03 0.5851 0.7617 0.4712


## 3. Các thành phần hybrid: chọn alpha, w_p, w_s trên valid (quy tắc một-SE)

In [5]:
def hybrid_terms(sp: str, alpha: float) -> tuple:
    return (
        smoothed_log(p_model[sp], alpha),
        smoothed_log(p_prior[sp], alpha),
        smoothed_log(same_dest_distribution(sd[sp]), alpha),
    )


def joint(r: np.ndarray) -> np.ndarray:
    """Per-row Recall@5 + NDCG@5 (float32)."""
    r = r.astype(np.float32)
    return ((r <= 5) * (1.0 + 1.0 / np.log2(r + 1))).astype(np.float32)


yv, uv = arr["valid"]["y"], arr["valid"]["user"]
t2 = time.time()
grid_rank = {}
for alpha in ALPHAS:
    M, P, S = hybrid_terms("valid", alpha)
    for wp, ws in itertools.product(W_GRID, W_GRID):
        grid_rank[(alpha, wp, ws)] = rank(M + wp * P + ws * S, yv).astype(np.int16)
print(f"{len(grid_rank)} configurations ranked on valid in {time.time() - t2:.0f}s")


def complexity(c: tuple) -> tuple:
    """Simpler = smaller largest weight, then smaller weight sum, then more smoothing."""
    return (max(c[1], c[2]), c[1] + c[2], -c[0])


t3 = time.time()
scores = {c: joint(r) for c, r in grid_rank.items()}
PICK, BEST, ELIGIBLE = one_se_pick(scores, uv, complexity)
del scores


def summ(c: tuple) -> dict:
    m = summarize(grid_rank[c])
    return {
        "alpha": c[0],
        "w_p": c[1],
        "w_s": c[2],
        "recall@5": m["recall@5"],
        "ndcg@5": m["ndcg@5"],
    }


print(f"best on valid {BEST}: {summ(BEST)}")
print(
    f"one-SE pick   {PICK}: {summ(PICK)}  ({len(ELIGIBLE)} configurations within one SE; {time.time() - t3:.0f}s)"
)
eligible_table = pd.DataFrame([summ(c) for c in sorted(ELIGIBLE, key=complexity)])
display(eligible_table)
best_per_alpha = {
    a: max([c for c in grid_rank if c[0] == a], key=lambda c: joint(grid_rank[c]).mean())
    for a in ALPHAS
}
print("best configuration per alpha (valid):")
display(pd.DataFrame([summ(c) for c in best_per_alpha.values()]))
ALPHA = PICK[0]

500 configurations ranked on valid in 108s


best on valid (0.1, 2.0, 2.5): {'alpha': 0.1, 'w_p': 2.0, 'w_s': 2.5, 'recall@5': 0.611, 'ndcg@5': 0.4479}
one-SE pick   (0.05, 2.0, 1.5): {'alpha': 0.05, 'w_p': 2.0, 'w_s': 1.5, 'recall@5': 0.611, 'ndcg@5': 0.4478}  (7 configurations within one SE; 11s)


,alpha,w_p,w_s,recall@5,ndcg@5
0,0.05,2.0,1.5,0.6110,0.4478
1,0.10,2.0,2.0,0.6110,0.4479
2,0.05,2.0,2.0,0.6109,0.4478
3,0.10,2.0,2.5,0.6110,0.4479
4,0.05,2.0,2.5,0.6109,0.4479
5,0.10,2.0,3.0,0.6110,0.4479
6,0.05,2.0,3.0,0.6109,0.4479


best configuration per alpha (valid):


,alpha,w_p,w_s,recall@5,ndcg@5
0,0.05,2.0,3.0,0.6109,0.4479
1,0.10,2.0,2.5,0.6110,0.4479
2,0.20,2.0,3.0,0.6105,0.4478
3,0.30,1.5,2.0,0.6104,0.4471
4,0.50,1.5,2.0,0.6093,0.4470


## 4. Cohort: cần bao nhiêu bộ trọng số? (CV ngoài fold trên valid, quy tắc một-SE trên các phương án)

Các phương án ứng viên chỉ dùng các mẫu cốt lõi có độ phủ rộng: độ dài lịch sử (L = 1 so với dài hơn) và việc destination truy vấn đã có trong lịch sử hay chưa ("known destination", các dòng mà số hạng sameDest hoạt động). Ở các dòng destination mới, số hạng sameDest là hằng số nên `w_s` được cố định 0 ở đó (không có tác dụng).

In [6]:
def schemes_for(a: dict) -> dict:
    """Cell id of every row under each candidate scheme."""
    L, qs = a["length"], a["q_seen"]
    return {
        "global (1 set)": np.zeros(len(L), int),
        "known vs new destination (2)": qs.astype(int),
        "L=1 vs L>=2 (2)": (L >= 2).astype(int),
        "L = 1 | 2-4 | 5+ (3)": np.digitize(L, [2, 5]),
        "4 L buckets (4)": np.digitize(L, [2, 5, 10]),
        "L=1 vs L>=2 x known/new (4)": (L >= 2) * 2 + qs,
    }


qs_v = arr["valid"]["q_seen"]
SCHEMES = schemes_for(arr["valid"])
cfg_a = [c for c in grid_rank if c[0] == ALPHA]
u_va, inv_va = np.unique(uv, return_inverse=True)
folds = {s: np.random.default_rng(s).integers(0, 2, len(u_va))[inv_va] for s in CV_SEEDS}


def pick_cell(mask: np.ndarray) -> tuple:
    cand = cfg_a if qs_v[mask].any() else [c for c in cfg_a if c[2] == 0.0]
    return one_se_pick({c: joint(grid_rank[c][mask]) for c in cand}, uv[mask], complexity)[0]


t4 = time.time()
oof = {}
for name, cell in SCHEMES.items():
    per_seed = []
    for s in CV_SEEDS:
        r_oof = np.zeros(len(yv), np.int64)
        for half in (0, 1):
            tune, ev = folds[s] != half, folds[s] == half
            for c_id in np.unique(cell):
                m = cell == c_id
                r_oof[m & ev] = grid_rank[pick_cell(m & tune)][m & ev]
        per_seed.append(joint(r_oof))
    oof[name] = np.mean(per_seed, axis=0)
print(f"out-of-fold CV over {len(SCHEMES)} schemes: {time.time() - t4:.0f}s")
n_cells = {name: len(np.unique(c)) for name, c in SCHEMES.items()}
SCHEME, BEST_SCHEME, ok_schemes = one_se_pick(oof, uv, lambda n: (n_cells[n],))
scheme_table = {}
for name, cell in SCHEMES.items():
    d = oof[name] - oof["global (1 set)"]
    scheme_table[name] = {
        "cells": n_cells[name],
        "min_coverage": round(float(np.bincount(cell).min() / len(cell)), 4),
        "oof_joint": round(float(oof[name].mean()), 5),
        "gain_vs_global": round(float(d.mean()), 5),
        "se_of_gain": round(cluster_se(d, uv), 5),
        "within_one_se_of_best": name in ok_schemes,
    }
display(pd.DataFrame(scheme_table).T)
print(f"best scheme out of fold: {BEST_SCHEME}; one-SE pick (fewest cells): {SCHEME}")

cell_v = SCHEMES[SCHEME]
final_w = {int(c_id): pick_cell(cell_v == c_id) for c_id in np.unique(cell_v)}
print("final weights per cell (alpha, w_p, w_s), tuned on the whole valid split:", final_w)

out-of-fold CV over 6 schemes: 28s


,cells,min_coverage,oof_joint,gain_vs_global,se_of_gain,within_one_se_of_best
global (1 set),1,1.0,1.05877,0.0,0.0,False
known vs new destination (2),2,0.2852,1.05907,0.0003,0.00025,True
L=1 vs L>=2 (2),2,0.1808,1.05866,-0.00011,0.00004,False
L = 1 | 2-4 | 5+ (3),3,0.1808,1.05861,-0.00016,0.00005,False
4 L buckets (4),4,0.1808,1.05827,-0.0005,0.00017,False
L=1 vs L>=2 x known/new (4),4,0.0248,1.05885,0.00007,0.00025,False


best scheme out of fold: known vs new destination (2); one-SE pick (fewest cells): known vs new destination (2)


final weights per cell (alpha, w_p, w_s), tuned on the whole valid split: {0: (0.05, 2.0, 0.0), 1: (0.05, 1.25, 1.0)}


## 5. Test (đọc một lần): hybrid vs SMLP4Rec + prior

In [7]:
HYB = "SMLP4Rec + prior + sameDest (this notebook)"
terms_of = {}
for sp in arr:
    M, P, S = hybrid_terms(sp, ALPHA)
    cid = schemes_for(arr[sp])[SCHEME]
    wp = np.array([final_w[c][1] for c in cid], np.float32)[:, None]
    ws = np.array([final_w[c][2] for c in cid], np.float32)[:, None]
    ranks[sp][HYB] = rank(M + wp * P + ws * S, arr[sp]["y"])
    if sp == "test":
        terms_of[HYB] = [M, wp * P, ws * S]

inter = pd.read_csv(INTER_DEST, sep="\t").sort_values(
    ["user_id:token", "timestamp:float"], kind="stable"
)
first = inter.drop_duplicates("user_id:token", keep="first")
windows = {
    "valid": (first["timestamp:float"] >= t_train_max) & (first["timestamp:float"] < t_valid_max),
    "test": first["timestamp:float"] >= t_valid_max,
}
cold_rank = {}
for sp, mask in windows.items():
    f = first[mask]
    y_c = f["item_id:token"].astype(int).values
    cold_rank[sp] = rank(
        np.log(prior_prob(f["srch_destination_id:float"].values.astype(np.int64)) + EPS), y_c
    )

COLS = ["recall@5", "ndcg@5", "recall@10", "ndcg@10", "recall@20", "ndcg@20"]
bench = {sp: {"warm": {}, "all_events": {}} for sp in arr}
for sp in arr:
    for name, r in ranks[sp].items():
        bench[sp]["warm"][name] = summarize(r)
        bench[sp]["all_events"][name] = summarize(np.concatenate([r, cold_rank[sp]]))
for sp in ("test", "valid"):
    for scope in ("warm", "all_events"):
        print(f"{sp.upper()}, {scope}")
        display(pd.DataFrame(bench[sp][scope]).T[COLS])

ra, rb = ranks["test"][HYB], ranks["test"][REF]
boot = {
    "hybrid minus SMLP4Rec + prior": {
        "warm": paired_bootstrap(ra, rb, k=5),
        "all_events": paired_bootstrap(
            np.concatenate([ra, cold_rank["test"]]), np.concatenate([rb, cold_rank["test"]]), k=5
        ),
    }
}
print("paired bootstrap, test (1000 resamples, seed 0, 95% CI)")
display(
    pd.DataFrame(
        {
            (k, scope): {
                m: f"{v[scope][m]['diff']:+.4f} [{v[scope][m]['ci95'][0]:+.4f}, {v[scope][m]['ci95'][1]:+.4f}]"
                for m in ("recall@5", "ndcg@5")
            }
            for k, v in boot.items()
            for scope in v
        }
    ).T
)

at = arr["test"]
slices = {
    "known destination": at["q_seen"],
    "new destination": ~at["q_seen"],
    "target already booked": at["old"],
    "target new to the user": ~at["old"],
    **bucket_masks["test"],
}


def r5_n5(r: np.ndarray, m: np.ndarray) -> str:
    v = summarize(r, m)
    return f"{v['recall@5']:.4f} / {v['ndcg@5']:.4f}"


slice_table = {
    s: {name: r5_n5(ranks["test"][name], m) for name in ranks["test"]} | {"rows": int(m.sum())}
    for s, m in slices.items()
}
print("TEST slices, Recall@5 / NDCG@5")
display(pd.DataFrame(slice_table).T)

TEST, warm


,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
"SMLP4Rec + prior (notebook 03, w_p = 1.5)",0.5851,0.4139,0.7617,0.4712,0.9073,0.5083
SMLP4Rec + prior + sameDest (this notebook),0.6028,0.4419,0.7701,0.4961,0.9093,0.5315


TEST, all_events


,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
"SMLP4Rec + prior (notebook 03, w_p = 1.5)",0.5715,0.4021,0.7504,0.4601,0.9014,0.4985
SMLP4Rec + prior + sameDest (this notebook),0.5858,0.4248,0.7573,0.4804,0.9030,0.5174


VALID, warm


...

,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
"SMLP4Rec + prior (notebook 03, w_p = 1.5)",0.5811,0.4088,0.7625,0.4676,0.9094,0.5050
SMLP4Rec + prior + sameDest (this notebook),0.5955,0.4314,0.7683,0.4875,0.9108,0.5237


paired bootstrap, test (1000 resamples, seed 0, 95% CI)


recall@5  \
hybrid minus SMLP4Rec + prior warm        +0.0177 [+0.0167, +0.0187]   
                              all_events  +0.0144 [+0.0136, +0.0152]   

                                                              ndcg@5  
hybrid minus SMLP4Rec + prior warm        +0.0280 [+0.0272, +0.0288]  
                              all_events  +0.0227 [+0.0221, +0.0234]

TEST slices, Recall@5 / NDCG@5


,"SMLP4Rec + prior (notebook 03, w_p = 1.5)",SMLP4Rec + prior + sameDest (this notebook),rows
known destination,0.6601 / 0.4947,0.7199 / 0.5875,60586
new destination,0.5554 / 0.3820,0.5565 / 0.3843,153283
target already booked,0.8085 / 0.6413,0.8657 / 0.7376,62430
target new to the user,0.4930 / 0.3201,0.4944 / 0.3200,151439
L=1,0.5500 / 0.3824,0.5569 / 0.3954,39778
L=2-4,0.5726 / 0.4022,0.5845 / 0.4228,67266
L=5-9,0.5959 / 0.4225,0.6136 / 0.4512,46340
L>=10,0.6137 / 0.4410,0.6449 / 0.4865,60485


## 6. Kiểm tra thành phần: mỗi thành phần có đóng góp trọng số không?

- **Spread share**: độ lệch chuẩn theo từng dòng giữa các cluster của mỗi số hạng có trọng số, tính theo tỷ lệ trong ba số hạng (ai quyết định thứ tự). Báo cáo trên các dòng mà destination truy vấn đã có trong lịch sử (cả ba số hạng hoạt động) và trên các dòng còn lại (sameDest là hằng số ở đó).
- **Drop-one**: Recall@5 / NDCG@5 trên test khi bỏ một số hạng và giữ nguyên các số hạng còn lại (giá trị của từng số hạng).

In [8]:
def spread_shares(terms: list, mask: np.ndarray) -> list:
    s = np.array([float(t[mask].std(1).mean()) for t in terms])
    return [round(float(v), 3) for v in s / s.sum()]


components = {}
for name, t in terms_of.items():
    full = rank(sum(t), at["y"])
    drops = {}
    for i, comp in enumerate(("SMLP4Rec", "prior", "sameDest")):
        r = rank(sum(x for j, x in enumerate(t) if j != i), at["y"])
        drops[f"drop {comp}"] = (
            f"{(r <= 5).mean() - (full <= 5).mean():+.4f} / {((r <= 5) / np.log2(r + 1)).mean() - ((full <= 5) / np.log2(full + 1)).mean():+.4f}"
        )
    components[name] = {
        "spread share M/P/S, known destination": spread_shares(t, at["q_seen"]),
        "spread share M/P/S, new destination": spread_shares(t, ~at["q_seen"]),
        **drops,
    }
display(pd.DataFrame(components).T)

,drop SMLP4Rec,drop prior,drop sameDest,"spread share M/P/S, known destination","spread share M/P/S, new destination"
SMLP4Rec + prior + sameDest (this notebook),-0.0231 / -0.0175,-0.2647 / -0.1871,-0.0146 / -0.0232,"[0.27, 0.52, 0.21]","[0.236, 0.764, 0.0]"


## 7. Lưu

In [9]:
result = {
    "formula": "score = log q_SMLP4Rec + w_p * log q_prior + w_s * log q_sameDest; log q = log((1 - alpha) * p + alpha / 100); p_sameDest = 0.7^age-weighted same-destination bookings normalized over clusters, uniform when none; cold rows (L = 0) prior only",
    "selection": {
        "objective": "per-row Recall@5 + NDCG@5 on valid",
        "rule": "one-standard-error rule, SE of the paired per-row difference clustered by user; simplest = smallest max weight, then smallest weight sum, then largest alpha",
        "grid": {"alpha": ALPHAS, "w_p": W_GRID, "w_s": W_GRID, "w_SMLP4Rec": 1.0},
        "best_on_valid": list(BEST),
        "one_se_pick": list(PICK),
        "n_within_one_se": len(ELIGIBLE),
        "eligible": [summ(c) for c in sorted(ELIGIBLE, key=complexity)],
        "best_per_alpha": [summ(c) for c in best_per_alpha.values()],
    },
    "cohorts": {
        "cv": "user-level 2-fold on valid, seeds "
        + str(CV_SEEDS)
        + "; cell weights by the one-SE rule on the tuning half",
        "schemes": scheme_table,
        "best_out_of_fold": BEST_SCHEME,
        "chosen_fewest_cells_within_one_se": SCHEME,
        "final_weights_per_cell": {str(k): list(v) for k, v in final_w.items()},
    },
    "reference": {"name": REF, "w_p": FUSION_W, "source": "smlprec_expedia_late_fusion.json"},
    "benchmark": bench,
    "test_slices_recall@5_ndcg@5": slice_table,
    "component_check_test": components,
    "bootstrap_test_95ci": boot,
}
for path in (RUN_DIR / "hybrid_samedest.json", REF_DIR / "smlprec_expedia_hybrid_samedest.json"):
    path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print("saved", REF_DIR / "smlprec_expedia_hybrid_samedest.json")

saved C:\workspace\PROJECTS\next-best-product-recommendation\results\week4_rebuild\clean_chain\smlprec_expedia_hybrid_samedest.json
